[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/therestiscode/notebooks/blob/main/04-one-nan/nan.ipynb)

# One missing value turned every number in my report into nan.

A latency export, five endpoints, one blank cell. NumPy finds the slowest endpoint
without a warning, and names the wrong one.

**The first eight code cells, `In [1]` to `In [8]`, are the ones in the short**, in the
same order, printing the same thing. After them, the rest of the evidence.

**Running this in Colab?** The file the first cell reads is not in your session yet.
Download [latency.csv](https://raw.githubusercontent.com/therestiscode/notebooks/main/04-one-nan/latency.csv),
open the folder icon on the left, and drop it in. Then run the cells in order.
Anywhere else, it already sits next to this notebook.

## The report

In [1]:
import numpy as np
rows = np.genfromtxt("latency.csv",
    delimiter=",", names=True,
    dtype=None, encoding="utf-8")
ms = rows["ms"]
names = rows["endpoint"]
print(ms.tolist())

[240.4, 190.2, 310.7, nan, 1205.9]


In [2]:
print(ms.argmax())

3


In [3]:
nan = ms[3]
print(nan > 1205.0)
print(nan < 1205.0)

False
False


In [4]:
print(nan == nan)

False


In [5]:
i = ms.argmax()
print(i, names[i])

3 /checkout


In [6]:
j = np.nanargmax(ms)
print(j, names[j])

4 /report


In [7]:
print(np.isnan(ms).any())

True


In [8]:
print(ms.max())

nan


---

## The rest of the evidence

`max`, `mean` and `median` all come back `nan`, with no warning, even when NumPy is
told to raise on every floating-point error:

In [9]:
print(ms.max(), ms.mean())
print(np.median(ms))

nan nan
nan


Why it is easy to miss: `nan` sorts to the end, so the first four values look perfect.
Print the head of the sorted array and the report looks correct.

In [10]:
print(np.sort(ms).tolist())

[190.2, 240.4, 310.7, 1205.9, nan]


And no warning, even when NumPy is told to raise on every floating-point error:

In [11]:
np.seterr(all="raise")
print(ms.max())

nan


`max` keeps a leader and swaps it when the next value is **not** `<=` the leader.
Every comparison with `nan` is `False`, so `nan` always gets swapped in, and nothing
swaps it back.

In [12]:
print(nan <= 1205.0)
print(np.isnan(ms).sum())

False
1


## The trap that is worse

Same file, whole milliseconds. With no `nan` to hold the blank, NumPy fills it with
the integer `-1`. `isnan` cannot find it, and the average is wrong without a word.

In [13]:
import io
whole = "endpoint,ms\n/login,240\n/search,190\n/cart,310\n/checkout,\n/report,1205\n"
irows = np.genfromtxt(io.StringIO(whole), delimiter=",",
    names=True, dtype=None, encoding="utf-8")
ims = irows["ms"]
print(ims.tolist(), ims.dtype)
print(ims.max())
print(ims.mean())
print(ims[ims != -1].mean())
print(np.isnan(ims.astype(float)).any())

[240, 190, 310, -1, 1205] int64
1205
388.8
486.25
False


---

[@therestiscode](https://youtube.com/@therestiscode)